# MILK10k — Step 10: Cost-efficient deployment benchmark
1. **Is MONET needed at inference?** MONET concept scores come precomputed in MILK10k, but in deployment they would need a large CLIP-based model per image. We retrain the fusion (late fusion + modality dropout, 5 folds) **with and without MONET inputs** and compare accuracy, set size and triage.
2. **Compression:** export the v2 clinical and dermoscopy networks (fold 0, seed 42) and the fusion head to **ONNX**, apply **INT8 dynamic quantisation**; report parameters, FLOPs, file size.
3. **Fidelity:** accuracy of FP32 PyTorch vs ONNX FP32 vs ONNX INT8 on all fold-0 test lesions, single view vs test-time augmentation.
4. **Latency:** preprocessing, each network (CPU 1 and 4 threads; GPU fp16), fusion + triage, and **expected time per patient** under selective dermoscopy.
5. **End-to-end check:** the deployed INT8 pipeline (two-stage, tone-calibrated conformal triage) against the reference pipeline on fold 0.
6. **Paper figure:** v2 dermoscopy-request curves, mean ± SD over seeds (from notebook 08 outputs).

**Inputs:** v2 seed-42 notebook, notebook08 output, MILK10_DATASET. **Settings:** GPU T4 (for the GPU rows; everything else runs on the CPU cores), **Internet On**. Runtime ≈ 30–45 min.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=False)
import os, io, time, copy, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from PIL import Image
from scipy.optimize import minimize_scalar
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import AutoModel

def find_dir(basename, must_have):
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if os.path.basename(root) == basename and must_have in files: return root
    return None
def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if name in files: return os.path.join(root, name)
    return None
RUN = find_dir("stage2v2_s42", "net_D_fold0.pt"); assert RUN, "attach the v2 seed-42 notebook"
les = pd.read_csv(f"{RUN}/milk10k_lesions_folds.csv")
IMG_DIR = os.path.dirname(find_file(les["isic_id_D"].iloc[0] + ".jpg")); assert IMG_DIR, "attach MILK10_DATASET"
OUT = "/kaggle/working/stage10"; os.makedirs(OUT, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| CPU cores:", os.cpu_count(), "| onnxruntime", ort.__version__, "\nrun:", RUN)

SEED = 42
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
y_all = y = les["y"].values.astype(int); fold = les["fold"].values; n = len(y)
MODS = ["C", "D", "M"]; PATTERNS = ["CDM", "CD", "CM", "DM", "C", "D", "M"]
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]; is_act_cls = np.isin(CLASSES, ACTIONABLE); y_act = is_act_cls[y]
calib_group = np.array([("MEL" if CLASSES[k] in ("MEL", "MAL_OTH") else CLASSES[k]) for k in y]); ACT_GROUPS = ["AKIEC", "BCC", "MEL", "SCCKA"]
tone = pd.cut(les["skin_tone_class"], [-1, 2, 3, 5], labels=["darker(0-2)", "medium(3)", "lighter(4-5)"]).astype(str).values
TONES = ["darker(0-2)", "medium(3)", "lighter(4-5)"]; eps = 1e-9
SIZE = {"C": 224, "D": 384}; BACKBONE = {"C": "facebook/convnextv2-tiny-22k-224", "D": "facebook/convnextv2-tiny-22k-384"}

## 1. Fusion with vs without MONET inputs (5 folds, code from notebook 03)

In [ ]:
MONET = [c[:-2] for c in les.columns if c.startswith("MONET_") and c.endswith("_C")]
site_d = pd.get_dummies(les["site"].fillna("unknown")).astype(np.float32)
SITE_COLS = list(site_d.columns)
meta_raw = np.column_stack([les["age_approx"].fillna(-1).values, les["age_approx"].isna().values, les["skin_tone_class"].values,
                            (les["sex"] == "male").values, site_d.values]).astype(np.float32)
EMB = {(m, f): np.load(f"{RUN}/emb_{m}_fold{f}.npy").astype(np.float32) for m in "CD" for f in range(5)}
USE_MONET = True
def fold_inputs(f, return_stats=False):
    tr_mask = fold != f; X, stats = {}, {}
    for m in ["C", "D"]:
        X[m] = EMB[(m, f)] if not USE_MONET else np.concatenate([EMB[(m, f)], les[[f"{c}_{m}" for c in MONET]].values.astype(np.float32)], 1)
    X["M"] = meta_raw.copy(); age = X["M"][:, 0]; age_med = np.median(age[tr_mask & (age >= 0)]); age[age < 0] = age_med
    for m in MODS:
        mu, sd = X[m][tr_mask].mean(0), X[m][tr_mask].std(0) + 1e-6; X[m] = (X[m] - mu) / sd; stats[m] = (mu, sd)
    stats["age_median"] = age_med
    return (X, stats) if return_stats else X
def backbone_val_ids(f):
    trval = les[les["fold"] != f]
    _, va = train_test_split(trval, test_size=0.10, stratify=trval["y"], random_state=SEED)
    return les.index.isin(va.index)

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, dims, kind="xattn", d=256, drop=0.3):
        super().__init__()
        self.kind = kind
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Dropout(0.2 if m != "M" else 0.0), nn.Linear(dims[m], d),
                                                    nn.GELU(), nn.LayerNorm(d)) for m in MODS})
        if kind == "late":
            self.heads = nn.ModuleDict({m: nn.Sequential(nn.Dropout(drop), nn.Linear(d, K)) for m in MODS})
        else:
            self.mod_emb = nn.Parameter(torch.randn(3, d) * 0.02)
            self.cls = nn.Parameter(torch.zeros(1, 1, d))
            layer = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=2 * d, dropout=0.1,
                                               batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, num_layers=2)
            self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, K))

    def forward(self, x, avail):                      # avail: (B,3) bool in MODS order
        tok = torch.stack([self.proj[m](x[m]) for m in MODS], 1)          # (B,3,d)
        if self.kind == "late":
            logits = torch.stack([self.heads[m](tok[:, i]) for i, m in enumerate(MODS)], 1)
            w = avail.float().unsqueeze(-1)
            return (logits * w).sum(1) / w.sum(1).clamp(min=1)
        tok = tok + self.mod_emb
        seq = torch.cat([self.cls.expand(len(tok), -1, -1), tok], 1)
        pad = torch.cat([torch.zeros(len(tok), 1, dtype=torch.bool, device=tok.device), ~avail], 1)
        return self.head(self.enc(seq, src_key_padding_mask=pad)[:, 0])

def pattern_mask(p, n):
    return torch.tensor([[m in p for m in MODS]] * n, dtype=torch.bool, device=device)

NONEMPTY = torch.tensor([[m in p for m in MODS] for p in PATTERNS], dtype=torch.bool)

def sample_avail(n, mode, fixed=None):
    """mode: 'fixed' (dedicated), 'none' (always full), 'dropout' (50% full, else a uniform non-empty subset)."""
    if mode == "fixed": return pattern_mask(fixed, n)
    full = torch.ones(n, 3, dtype=torch.bool)
    if mode == "none": return full.to(device)
    sub = NONEMPTY[torch.randint(len(PATTERNS), (n,))]
    keep_full = torch.rand(n) < 0.5
    return torch.where(keep_full.unsqueeze(1), full, sub).to(device)

In [ ]:
def evaluate(y, P):
    pred = P.argmax(1)
    out = {"bal_acc": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
           "macro_f1_thr05": np.mean([f1_score(y == k, P[:, k] >= 0.5, zero_division=0) for k in range(K)])}
    try: out["macro_auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=range(K))
    except ValueError: out["macro_auc"] = np.nan
    return out

def to_t(X, idx): return {m: torch.tensor(X[m][idx], device=device) for m in MODS}

@torch.no_grad()
def predict(model, X, idx, pattern):
    model.eval(); x = to_t(X, idx)
    return model(x, pattern_mask(pattern, len(idx))).float().softmax(1).cpu().numpy()

def train_fusion(X, f, kind, mode, fixed=None, epochs=80, patience=15, bs=256, lr=3e-4):
    torch.manual_seed(SEED + f); np.random.seed(SEED + f)
    is_val = backbone_val_ids(f)
    tr_idx = np.where((les["fold"].values != f) & ~is_val)[0]
    va_idx = np.where(is_val)[0]
    val_pattern = fixed if mode == "fixed" else "CDM"
    n = np.bincount(y_all[tr_idx], minlength=K).astype(float)
    w = np.sqrt(n.max() / np.maximum(n, 1)); w = torch.tensor(w / w.mean(), dtype=torch.float, device=device)
    model = FusionNet({m: X[m].shape[1] for m in MODS}, kind).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    Xtr, ytr = to_t(X, tr_idx), torch.tensor(y_all[tr_idx], device=device)
    best, bad, state = -1, 0, None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr_idx), device=device)
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            xb = {m: Xtr[m][b] for m in MODS}
            loss = F.cross_entropy(model(xb, sample_avail(len(b), mode, fixed)), ytr[b],
                                   weight=w, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step()
        # early stopping on out-of-sample lesions; with dropout, score = mean over all 7 patterns
        pats = [val_pattern] if mode != "dropout" else PATTERNS
        score = np.mean([evaluate(y_all[va_idx], predict(model, X, va_idx, p))["macro_f1"] for p in pats])
        if score > best: best, bad, state = score, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state)
    return model, ep + 1

In [ ]:
VAR = {}
for use_monet in [True, False]:
    USE_MONET = use_monet; probs = {p: np.zeros((n, K), np.float32) for p in PATTERNS}; t0 = time.time()
    for f in range(5):
        X, stats = fold_inputs(f, return_stats=True); te = np.where(fold == f)[0]
        model, _ = train_fusion(X, f, kind="late", mode="dropout")
        for p in PATTERNS: probs[p][te] = predict(model, X, te, p)
        if f == 0: fold0_model, fold0_stats = copy.deepcopy(model).cpu().eval(), stats
    VAR["with MONET" if use_monet else "without MONET"] = dict(probs=probs, model=fold0_model, stats=fold0_stats)
    print(("with" if use_monet else "without"), f"MONET: fusion trained on 5 folds in {time.time() - t0:.0f}s")

In [ ]:
def temp(P, T):
    L = np.log(P + eps) / T; L -= L.max(1, keepdims=True); E = np.exp(L); return E / E.sum(1, keepdims=True)
def nll(P, yy): return -np.log(P[np.arange(len(yy)), yy] + eps).mean()
def calibrate(P):
    out = np.zeros_like(P, dtype=float)
    for f in range(5):
        tr, te = fold != f, fold == f
        T = minimize_scalar(lambda t: nll(temp(P[tr], t), y[tr]), bounds=(0.2, 5), method="bounded").x
        out[te] = temp(P[te], T)
    return out
def q_upper(v, a):
    v = np.sort(v); m = len(v); k = int(np.ceil((m + 1) * (1 - a)))
    return np.inf if (m == 0 or k > m) else v[k - 1]
def set_size(P, alpha=0.10):
    S = np.zeros((n, K), bool); st = 1 - P[np.arange(n), y]
    for f in range(5):
        cal, te = fold != f, fold == f; S[te] = (1 - P[te]) <= q_upper(st[cal], alpha)
    return S.sum(1).mean(), S[np.arange(n), y].mean()

# ---- policy ----
TARGET_SPEC = 0.70; RATES = np.round(np.arange(0, 1.0001, 0.05), 2)
def act_score(P): return P[:, is_act_cls].sum(1)
def spec_thresholds(P):
    t = np.zeros(n)
    for f in range(5): t[fold == f] = np.quantile(act_score(P)[(fold != f) & ~y_act], TARGET_SPEC)
    return t
def policy_metrics(P_CM, P_CDM, thr_CM, thr_CDM, req):
    P = np.where(req[:, None], P_CDM, P_CM); flag = act_score(P) >= np.where(req, thr_CDM, thr_CM)
    return {"macro_f1": f1_score(y, P.argmax(1), average="macro", labels=range(K), zero_division=0),
            "act_sens@spec": (flag & y_act).sum() / y_act.sum()}
def run_policy(P_CM, P_CDM):
    tCM, tCDM = spec_thresholds(P_CM), spec_thresholds(P_CDM); top2 = np.sort(P_CM, 1)[:, -2:]
    scores = {"random": np.random.RandomState(0).rand(n), "least_conf": 1 - P_CM.max(1),
              "entropy": -(P_CM * np.log(P_CM + eps)).sum(1), "small_margin": -(top2[:, 1] - top2[:, 0])}
    rows = []
    for name, s in scores.items():
        order = np.argsort(-s)
        for q in RATES:
            req = np.zeros(n, bool); req[order[:int(round(q * n))]] = True
            rows.append(dict(policy=name, rate=q, **policy_metrics(P_CM, P_CDM, tCM, tCDM, req)))
    cur = pd.DataFrame(rows); out = {}
    for m in ["macro_f1", "act_sens@spec"]:
        lo, hi = cur.loc[cur.rate == 0, m].iloc[0], cur.loc[cur.rate == 1, m].iloc[0]; tgt = lo + 0.95 * (hi - lo)
        for name, d in cur.groupby("policy"):
            ok = d[d[m] >= tgt]; out[f"rate95_{m}_{name}"] = ok["rate"].min() if len(ok) else np.nan
        out[f"{m}_CM_only"], out[f"{m}_CDM_all"] = lo, hi
        out[f"{m}_least_conf@30%"] = cur[(cur.policy == "least_conf") & (cur.rate == 0.30)][m].iloc[0]
    return out, cur

# ---- conformal triage (two-stage, binary) ----
def tri_sets(P, cal_fn, aA, aB, variant):
    s = act_score(P); inA, inB = np.zeros(n, bool), np.zeros(n, bool)
    for f in range(5):
        te, cal = fold == f, cal_fn(f)
        inB[te] = s[te] <= q_upper(s[cal & ~y_act], aB)
        if variant == "group":
            inA[te] = (1 - s[te]) <= q_upper(1 - s[cal & y_act], aA)
        elif variant == "worst_class":
            inA[te] = (1 - s[te]) <= max(q_upper(1 - s[cal & (calib_group == g)], aA) for g in ACT_GROUPS)
        elif variant == "tone":
            for g in TONES:
                m = te & (tone == g); inA[m] = (1 - s[m]) <= q_upper(1 - s[cal & y_act & (tone == g)], aA)
    return inA, inB
def decide(inA, inB): return np.where(inA & ~inB, "refer", np.where(inB & ~inA, "reassure", "ambiguous"))
def triage(P_CM, P_CDM, alpha, aB, strategy, variant):
    if strategy == "all":
        d = decide(*tri_sets(P_CDM, lambda f: fold != f, alpha, aB, variant)); return np.where(d == "ambiguous", "refer", d), np.ones(n, bool)
    aA = alpha / 2 if strategy == "adaptive" else alpha
    d1 = decide(*tri_sets(P_CM, lambda f: fold != f, aA, aB, variant))
    if strategy == "none": return np.where(d1 == "ambiguous", "refer", d1), np.zeros(n, bool)
    req = d1 == "ambiguous"; d2 = decide(*tri_sets(P_CDM, lambda f: (fold != f) & req, aA, aB, variant))
    final = d1.copy(); final[req] = np.where(d2[req] == "ambiguous", "refer", d2[req]); return final, req
def tri_summary(final, req):
    re = final == "reassure"; dk = tone == "darker(0-2)"; mel = calib_group == "MEL"
    return {"dermoscopy_rate": req.mean(), "act_missed": (re & y_act).sum() / y_act.sum(),
            "MEL_missed": (re & mel).sum() / mel.sum(), "darker_missed": (re & y_act & dk).sum() / (y_act & dk).sum(),
            "benign_reassured": (re & ~y_act).sum() / (~y_act).sum(), "referral_rate": (final == "refer").mean()}

In [ ]:
rows = {}
for name, v in VAR.items():
    P = v["probs"]; CAL = {p: calibrate(P[p].astype(float)) for p in ["CM", "CDM"]}; v["CAL"] = CAL
    r = {}
    for p in ["CDM", "CM"]:
        e = evaluate(y, P[p]); r[f"{p} macro F1"], r[f"{p} AUC"] = e["macro_f1"], e["macro_auc"]
        r[f"{p} set size (90%)"] = set_size(CAL[p])[0]
    for k_, val in tri_summary(*triage(CAL["CM"], CAL["CDM"], 0.10, 0.20, "adaptive", "tone")).items(): r[f"triage (tone) {k_}"] = val
    rows[name] = r
cmp = pd.DataFrame(rows); cmp["difference"] = cmp["without MONET"] - cmp["with MONET"]
cmp.to_csv(f"{OUT}/T1_monet_ablation.csv")
DEPLOY = "without MONET" if (cmp.loc["CDM macro F1", "difference"] > -0.01 and cmp.loc["CM macro F1", "difference"] > -0.01) else "with MONET"
print("Deployed fusion variant:", DEPLOY, "(MONET is dropped if macro F1 falls by less than 0.01)")
cmp.round(4)

## 2. Export: ONNX FP32 and INT8, parameters, FLOPs, file size

In [ ]:
class Net(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.backbone.config.hidden_sizes[-1], K))
    def embed(self, x): return self.backbone(pixel_values=x).pooler_output
    def forward(self, x): return self.head(self.embed(x))
class ExportNet(nn.Module):                     # outputs class probabilities and the 768-d embedding used by the fusion head
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, image):
        e = self.net.embed(image); return self.net.head(e).softmax(1), e
NETS = {}
for m in "CD":
    net = Net(BACKBONE[m]); net.load_state_dict(torch.load(f"{RUN}/net_{m}_fold0.pt", map_location="cpu")); NETS[m] = net.eval()

def export(module, args, path, input_names, output_names, dyn):
    kw = dict(input_names=input_names, output_names=output_names, dynamic_axes=dyn, opset_version=17, do_constant_folding=True)
    try: torch.onnx.export(module, args, path, dynamo=False, **kw)
    except TypeError: torch.onnx.export(module, args, path, **kw)
    onnx.checker.check_model(onnx.load(path))
    module.eval()   # torch.onnx.export restores the wrapper's train mode, which switches dropout back on

from torch.utils.flop_counter import FlopCounterMode
spec = []
for m in "CD":
    s = SIZE[m]; x = torch.randn(1, 3, s, s); fp32, int8 = f"{OUT}/net_{m}_fp32.onnx", f"{OUT}/net_{m}_int8.onnx"
    export(ExportNet(NETS[m]), (x,), fp32, ["image"], ["probs", "embedding"], {"image": {0: "batch"}, "probs": {0: "batch"}, "embedding": {0: "batch"}})
    quantize_dynamic(fp32, int8, weight_type=QuantType.QInt8)
    with FlopCounterMode(display=False) as fc, torch.no_grad(): NETS[m](x)
    spec.append(dict(network={"C": "clinical (224 px)", "D": "dermoscopy (384 px)"}[m],
                     params_M=sum(p.numel() for p in NETS[m].parameters()) / 1e6, GFLOPs=fc.get_total_flops() / 1e9,
                     onnx_fp32_MB=os.path.getsize(fp32) / 1e6, onnx_int8_MB=os.path.getsize(int8) / 1e6))

class FusionWrap(nn.Module):
    def __init__(self, fm): super().__init__(); self.fm = fm
    def forward(self, c, d, m, avail): return self.fm({"C": c, "D": d, "M": m}, avail > 0.5).softmax(1)
fm = VAR[DEPLOY]["model"]; dims = {k: VAR[DEPLOY]["stats"][k][0].shape[0] for k in MODS}
fargs = (torch.randn(1, dims["C"]), torch.randn(1, dims["D"]), torch.randn(1, dims["M"]), torch.ones(1, 3))
export(FusionWrap(fm), fargs, f"{OUT}/fusion_fp32.onnx", ["c", "d", "m", "avail"], ["probs"],
       {k: {0: "batch"} for k in ["c", "d", "m", "avail", "probs"]})
spec.append(dict(network=f"fusion head ({DEPLOY})", params_M=sum(p.numel() for p in fm.parameters()) / 1e6,
                 GFLOPs=np.nan, onnx_fp32_MB=os.path.getsize(f"{OUT}/fusion_fp32.onnx") / 1e6, onnx_int8_MB=np.nan))
for _n in NETS.values(): _n.eval()
fm.eval()
spec = pd.DataFrame(spec).set_index("network"); spec.to_csv(f"{OUT}/T2_model_spec.csv")
print("GFLOPs = floating-point operations per image (2 x multiply-accumulates)."); spec.round(2)

## 3. Fidelity on all fold-0 test lesions: PyTorch FP32 vs ONNX FP32 vs ONNX INT8 (CPU), single view vs TTA

In [ ]:
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
def shades_of_gray(a, p=6):
    a = a.astype(np.float32); il = np.power(np.mean(np.power(a, p), axis=(0, 1)), 1 / p)
    il = il / (np.sqrt((il ** 2).sum()) + 1e-6); return np.clip(a / (il * np.sqrt(3) + 1e-6), 0, 255).astype(np.uint8)
def preprocess_uint8(path, m):
    """Deployment preprocessing = training preprocessing: short-side resize, Shades-of-Gray, square resize."""
    s = SIZE[m]; short = int(s * 1.15); im = Image.open(path).convert("RGB"); w_, h_ = im.size; r = short / min(w_, h_)
    im = im.resize((max(short, round(w_ * r)), max(short, round(h_ * r))), Image.BICUBIC)
    return np.asarray(Image.fromarray(shades_of_gray(np.asarray(im))).resize((s, s), Image.BILINEAR))
def to_input(u8): return ((u8.astype(np.float32) / 255 - MEAN) / STD).transpose(0, 3, 1, 2).copy()
# raw MILK10k images only: skip the "cache" folders that training notebooks leave in their output
RAW = {}
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if "cache" in root.lower(): continue
    for q in files:
        if q.lower().endswith(".jpg"): RAW.setdefault(q[:-4], os.path.join(root, q))
need = set(les["isic_id_C"]) | set(les["isic_id_D"]); miss = need - set(RAW)
assert not miss, f"{len(miss)} raw images not found (e.g. {list(miss)[:2]}): attach MILK10_DATASET"
def img_path(i, m): return RAW[les[f"isic_id_{m}"].iloc[i]]

te0 = np.where(fold == 0)[0]; t0 = time.time()
U8 = {m: np.stack([preprocess_uint8(img_path(i, m), m) for i in te0]) for m in "CD"}
print(f"preprocessed {len(te0)} lesions x 2 images in {time.time() - t0:.0f}s")

def run_torch(m, tta):
    net = NETS[m].to(device); out_p, out_e = [], []
    with torch.no_grad():
        for b in range(0, len(te0), 32):
            x = torch.tensor(to_input(U8[m][b:b + 32]), device=device)
            if tta: views = ([torch.rot90(v, k, dims=(2, 3)) for v in (x, torch.flip(x, dims=[3])) for k in range(4)] if m == "D" else [x, torch.flip(x, dims=[3])])
            else: views = [x]
            e = torch.stack([net.embed(v) for v in views]).mean(0); out_p.append(net.head(e).softmax(1).cpu().numpy()); out_e.append(e.cpu().numpy())
    NETS[m].cpu(); return np.concatenate(out_p), np.concatenate(out_e)
def run_ort(path, m, threads=os.cpu_count()):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"]); out_p, out_e = [], []
    for b in range(0, len(te0), 16):
        p_, e_ = sess.run(None, {"image": to_input(U8[m][b:b + 16])}); out_p.append(p_); out_e.append(e_)
    return np.concatenate(out_p), np.concatenate(out_e)

RES = {}
for m in "CD":
    t0 = time.time()
    RES[(m, "PyTorch FP32, TTA")] = run_torch(m, True)
    RES[(m, "PyTorch FP32, single view")] = run_torch(m, False)
    RES[(m, "ONNX FP32, single view")] = run_ort(f"{OUT}/net_{m}_fp32.onnx", m)
    RES[(m, "ONNX INT8, single view")] = run_ort(f"{OUT}/net_{m}_int8.onnx", m)
    print(m, f"done in {time.time() - t0:.0f}s")
yt = y[te0]; rows = []
for (m, cfg), (Pp, E) in RES.items():
    ref = RES[(m, "PyTorch FP32, single view")][0]; e = evaluate(yt, Pp)
    rows.append(dict(network={"C": "clinical", "D": "dermoscopy"}[m], config=cfg, macro_f1=e["macro_f1"], macro_auc=e["macro_auc"],
                     bal_acc=e["bal_acc"], MEL_recall=recall_score(yt == 7, Pp.argmax(1) == 7),
                     top1_agreement_vs_fp32=(Pp.argmax(1) == ref.argmax(1)).mean(), max_abs_prob_diff=np.abs(Pp - ref).max()))
fid = pd.DataFrame(rows).set_index(["network", "config"]); fid.to_csv(f"{OUT}/T3_fidelity.csv"); fid.round(4)

## 4. Latency (median of 30 runs, batch 1) and expected time per patient

In [ ]:
def ort_latency(path, feeds, threads, reps=30):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads; so.inter_op_num_threads = 1
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    for _ in range(5): sess.run(None, feeds)
    ts = []
    for _ in range(reps): t = time.perf_counter(); sess.run(None, feeds); ts.append(time.perf_counter() - t)
    return 1000 * np.median(ts)
lat = []
for m in "CD":
    paths = [img_path(i, m) for i in te0[:30]]
    ts = []
    for p in paths: t = time.perf_counter(); preprocess_uint8(p, m); ts.append(time.perf_counter() - t)
    lat.append(dict(stage=f"preprocess {m} (decode + resize + colour constancy)", engine="CPU, 1 thread", ms=1000 * np.median(ts)))
    feeds = {"image": to_input(U8[m][:1])}
    for prec in ["fp32", "int8"]:
        for th in [1, 4]:
            lat.append(dict(stage=f"network {m}", engine=f"ONNX {prec.upper()}, CPU {th} thread{'s' if th > 1 else ''}",
                            ms=ort_latency(f"{OUT}/net_{m}_{prec}.onnx", feeds, th)))
    if device.type == "cuda":
        net = NETS[m].to(device).half(); x1 = torch.tensor(to_input(U8[m][:1]), device=device).half(); x32 = x1.repeat(32, 1, 1, 1)
        with torch.no_grad():
            for _ in range(5): net(x1)
            torch.cuda.synchronize(); ts = []
            for _ in range(30): t = time.perf_counter(); net(x1); torch.cuda.synchronize(); ts.append(time.perf_counter() - t)
            for _ in range(3): net(x32)
            torch.cuda.synchronize(); t = time.perf_counter()
            for _ in range(10): net(x32)
            torch.cuda.synchronize(); thr = 320 / (time.perf_counter() - t)
        lat.append(dict(stage=f"network {m}", engine="PyTorch FP16, GPU (T4), batch 1", ms=1000 * np.median(ts), throughput_img_s=thr))
        NETS[m].float().cpu()
fd = {k: v.numpy() for k, v in zip(["c", "d", "m", "avail"], fargs)}
lat.append(dict(stage="fusion head + conformal decision", engine="ONNX FP32, CPU 1 thread", ms=ort_latency(f"{OUT}/fusion_fp32.onnx", fd, 1)))
lat = pd.DataFrame(lat); lat.to_csv(f"{OUT}/T4_latency.csv", index=False); lat.round(2)

In [ ]:
# expected time per patient: clinical path always; dermoscopy path only for lesions sent for dermoscopy
p_req = tri_summary(*triage(VAR[DEPLOY]["CAL"]["CM"], VAR[DEPLOY]["CAL"]["CDM"], 0.10, 0.20, "adaptive", "tone"))["dermoscopy_rate"]
def L(stage, engine): return lat[(lat.stage == stage) & (lat.engine == engine)]["ms"].iloc[0]
rows = []
for eng in ["ONNX FP32, CPU 4 threads", "ONNX INT8, CPU 4 threads", "ONNX INT8, CPU 1 thread"]:
    pre_C, pre_D = L("preprocess C (decode + resize + colour constancy)", "CPU, 1 thread"), L("preprocess D (decode + resize + colour constancy)", "CPU, 1 thread")
    fus = L("fusion head + conformal decision", "ONNX FP32, CPU 1 thread")
    stage1 = pre_C + L("network C", eng) + fus; stage2 = pre_D + L("network D", eng) + fus
    rows.append(dict(engine=eng, stage1_clinical_ms=stage1, stage2_dermoscopy_ms=stage2, dermoscopy_rate=p_req,
                     expected_ms_per_patient=stage1 + p_req * stage2, dermoscopy_for_all_ms=stage1 + stage2))
pp = pd.DataFrame(rows).set_index("engine"); pp.to_csv(f"{OUT}/T5_time_per_patient.csv"); pp.round(1)

## 5. End-to-end check on fold 0: deployed INT8 pipeline vs reference (two-stage, tone-calibrated conformal triage, α = 0.10, α_B = 0.20)

In [ ]:
V = VAR[DEPLOY]; stats = V["stats"]; fmod = V["model"].to(device)
def fusion_inputs(E_C, E_D, idx):
    feats = {}
    for m, E in [("C", E_C), ("D", E_D)]:
        Z = E if DEPLOY == "without MONET" else np.concatenate([E, les[[f"{c}_{m}" for c in MONET]].values[idx].astype(np.float32)], 1)
        feats[m] = (Z - stats[m][0]) / stats[m][1]
    Mm = meta_raw[idx].copy(); Mm[Mm[:, 0] < 0, 0] = stats["age_median"]; feats["M"] = (Mm - stats["M"][0]) / stats["M"][1]
    return {k: torch.tensor(v, dtype=torch.float32, device=device) for k, v in feats.items()}
def fuse(feats, pattern):
    with torch.no_grad(): return fmod(feats, pattern_mask(pattern, len(feats["C"]))).softmax(1).cpu().numpy()

# temperatures and thresholds from the calibration folds (1-4) only
def fit_T(P, mask): return minimize_scalar(lambda t: nll(temp(P[mask], t), y[mask]), bounds=(0.2, 5), method="bounded").x
cal = fold != 0; RAW = V["probs"]; T_ = {p: fit_T(RAW[p].astype(float), cal) for p in ["CM", "CDM"]}
PC = {p: temp(RAW[p].astype(float), T_[p]) for p in ["CM", "CDM"]}
aA, aB = 0.10 / 2, 0.20
def thresholds(s, mask):
    return q_upper(s[mask & ~y_act], aB), {g: q_upper(1 - s[mask & y_act & (tone == g)], aA) for g in TONES}
def decide_with(s, idx, th):
    qB, qA = th; inB = s <= qB; inA = np.array([(1 - s_) <= qA[tone[i]] for s_, i in zip(s, idx)])
    return np.where(inA & ~inB, "refer", np.where(inB & ~inA, "reassure", "ambiguous"))
sCM, sCDM = act_score(PC["CM"]), act_score(PC["CDM"])
th1 = thresholds(sCM, cal); cal_idx = np.where(cal)[0]
amb_cal = np.zeros(n, bool); amb_cal[cal_idx] = decide_with(sCM[cal_idx], cal_idx, th1) == "ambiguous"
th2 = thresholds(sCDM, cal & amb_cal)

def triage_fold0(P_CM0, P_CDM0):
    d1 = decide_with(act_score(P_CM0), te0, th1); req = d1 == "ambiguous"; final = d1.copy()
    d2 = decide_with(act_score(P_CDM0), te0, th2); final[req] = np.where(d2[req] == "ambiguous", "refer", d2[req]); return final, req
def summ0(final, req, P_CDM0):
    re = final == "reassure"; ya = y_act[te0]; mel = calib_group[te0] == "MEL"; dk = tone[te0] == "darker(0-2)"
    return {"dermoscopy_rate": req.mean(), "act_missed": (re & ya).sum() / ya.sum(), "MEL_missed": (re & mel).sum() / mel.sum(),
            "darker_missed": (re & ya & dk).sum() / max((ya & dk).sum(), 1), "benign_reassured": (re & ~ya).sum() / (~ya).sum(),
            "CDM macro F1": f1_score(y[te0], P_CDM0.argmax(1), average="macro", labels=range(K), zero_division=0)}

ref_final, ref_req = triage_fold0(PC["CM"][te0], PC["CDM"][te0])          # reference: TTA embeddings, PyTorch
outs = {"reference (PyTorch FP32, TTA)": (ref_final, ref_req, PC["CDM"][te0])}
for cfg in ["ONNX FP32, single view", "ONNX INT8, single view"]:
    feats = fusion_inputs(RES[("C", cfg)][1], RES[("D", cfg)][1], te0)
    P_CM0, P_CDM0 = temp(fuse(feats, "CM"), T_["CM"]), temp(fuse(feats, "CDM"), T_["CDM"])
    fin, rq = triage_fold0(P_CM0, P_CDM0); outs[f"deployed ({cfg})"] = (fin, rq, P_CDM0)
rows = {}
for name, (fin, rq, P_) in outs.items():
    r = summ0(fin, rq, P_); r["decision agreement with reference"] = (fin == ref_final).mean(); rows[name] = r
e2e = pd.DataFrame(rows).T; e2e.to_csv(f"{OUT}/T6_end_to_end_fold0.csv"); e2e.round(4)

## 6. Paper figure: v2 dermoscopy-request curves, mean ± SD over the three seeds (from notebook 08)

In [ ]:
curves = {}
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    b = os.path.basename(root)
    if "policy_curves.csv" in files and (b.startswith("v2_") or b.startswith("v1_")): curves.setdefault(b, pd.read_csv(f"{root}/policy_curves.csv"))
print("runs found:", sorted(curves))
v2 = [k for k in curves if k.startswith("v2_")]
if v2:
    allc = pd.concat([curves[k].assign(run=k) for k in v2])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
    for a, met, lab in zip(ax, ["macro_f1", "act_sens@spec"], ["macro F1 (11 classes)", "sensitivity for treatable lesions at 70% specificity"]):
        for pol, col, ls in [("least_conf", "C3", "-"), ("entropy", "C0", "-"), ("small_margin", "C2", "-"), ("random", "gray", "--")]:
            g = allc[allc.policy == pol].groupby("rate")[met]; mu, sd = g.mean(), g.std()
            a.plot(mu.index * 100, mu.values, ls, color=col, label=pol.replace("_", " ")); a.fill_between(mu.index * 100, mu - sd, mu + sd, color=col, alpha=0.15)
        a.set_xlabel("% of lesions sent for dermoscopy"); a.set_ylabel(lab); a.grid(alpha=0.3)
    ax[0].legend(); plt.tight_layout(); plt.savefig(f"{OUT}/F_policy_curves_v2.png", dpi=220); plt.show()
    allc.groupby(["policy", "rate"])[["macro_f1", "act_sens@spec"]].agg(["mean", "std"]).to_csv(f"{OUT}/T7_policy_curves_v2.csv")
else:
    print("Attach the committed notebook08 output to draw this figure.")